# Notebook 23 ResNet50 Baseline

**ResNet-50 whole-image baseline (reticular + ground-glass).**

ResNet50 baseline for the clean-vs-noisy detection task in exp07.

Replaces the raw-pixel feature with a 2048-d feature vector from an
ImageNet-pretrained ResNet50 (avgpool layer). Trains the same logistic
probe used by exp07. Only computes the perturbations referenced in the
manuscript: reticular_fine_p32 and ground_glass_p64.

Output rows are appended to the existing exp07 parquet with mode='resnet50'.

Usage:
    DATASET=nih python compute_resnet50_baseline.py


In [ ]:
# === Papermill parameters (override via `papermill -p NAME VALUE`) ===
DATASET = "nih"            # one of: nih, mimic, emory, vindr (where applicable)
SEED = 42
OUTPUTS_DIR = "/home/saptpurk/embeddings-noise-eliminators/outputs"
REPO_ROOT_OVERRIDE = "/home/saptpurk/embeddings-noise-eliminators"
HF_TOKEN_OVERRIDE = None     # set non-None when running gated models locally


In [ ]:
# Apply papermill parameters to environment
import os
os.environ.setdefault("DATASET", DATASET)
os.environ.setdefault("OUTPUTS_DIR", OUTPUTS_DIR)
os.environ.setdefault("REPO_ROOT", REPO_ROOT_OVERRIDE)
if HF_TOKEN_OVERRIDE:
    os.environ["HF_TOKEN"] = HF_TOKEN_OVERRIDE


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as tvm
import torchvision.transforms as T
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_recall_curve, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

REPO_ROOT = Path(os.environ.get(
    "REPO_ROOT", "/home/saptpurk/embeddings-noise-eliminators"))
sys.path.insert(0, str(REPO_ROOT))

from common import (
    PARAMS,
    GroundGlassInjector,
    ReticularPatternInjector,
    get_config,
    load_and_pad,
    load_disease_labels,
    stratified_split,
)

DATASET = os.environ.get("DATASET", "nih")
WORK = Path(os.environ.get(
    "OUTPUTS_DIR", "/home/saptpurk/embeddings-noise-eliminators/outputs"))
SEED = PARAMS.random_seed

# Same sample size and target perturbations as exp07.
N_TARGET = 20_000
PERTURBATIONS = [
    ("reticular_fine_p32",
     ReticularPatternInjector(seed=SEED, period_px=3,
                              amplitude=PARAMS.reticular_amplitude),
     32),
    ("ground_glass_p64",
     GroundGlassInjector(seed=SEED, sigma_px=PARAMS.ground_glass_sigma,
                         amplitude=PARAMS.ground_glass_amplitude),
     64),
]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESNET_INPUT = 224


def build_resnet50():
    weights = tvm.ResNet50_Weights.IMAGENET1K_V2
    model = tvm.resnet50(weights=weights)
    model.fc = nn.Identity()  # 2048-d avgpool features
    model.eval().to(DEVICE)
    return model, weights.transforms()


def preprocess(img, transform):
    """img is HxW grayscale uint8 (0..255). Convert to 3xHxW float tensor."""
    if img.ndim == 2:
        img = np.repeat(img[:, :, None], 3, axis=2)
    elif img.shape[2] == 1:
        img = np.repeat(img, 3, axis=2)
    img = torch.from_numpy(img).permute(2, 0, 1).contiguous()
    img = T.functional.resize(img, [RESNET_INPUT, RESNET_INPUT], antialias=True)
    img = img.float() / 255.0
    # Manual ImageNet normalization (transform from weights expects PIL):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img = (img - mean) / std
    return img


@torch.no_grad()
def extract_features(model, df, injector, patch_size, batch_size=32):
    feats_clean, feats_pert = [], []
    batch_clean, batch_pert = [], []

    def _flush():
        if batch_clean:
            xc = torch.stack(batch_clean).to(DEVICE, non_blocking=True)
            xp = torch.stack(batch_pert).to(DEVICE, non_blocking=True)
            feats_clean.append(model(xc).cpu().numpy())
            feats_pert.append(model(xp).cpu().numpy())
            batch_clean.clear()
            batch_pert.clear()

    for _, row in tqdm(df.iterrows(), total=len(df),
                       desc=f"resnet50 {injector.__class__.__name__}/{patch_size}"):
        clean = load_and_pad(row["image_path"], CFG.target_size)
        noisy, _ = injector(clean, patch_size=patch_size, num_patches=1,
                            image_path=row["image_path"])
        batch_clean.append(preprocess(clean, None))
        batch_pert.append(preprocess(noisy, None))
        if len(batch_clean) >= batch_size:
            _flush()
    _flush()
    return np.vstack(feats_clean), np.vstack(feats_pert)


def fit_eval(X_train, y_train, X_test, y_test, n_boot=PARAMS.n_bootstrap):
    scaler = StandardScaler().fit(X_train)
    Xtr = scaler.transform(X_train)
    Xte = scaler.transform(X_test)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    best_C, best_cv = None, -np.inf
    for C in PARAMS.lr_C_grid:
        cv_aucs = []
        for tr_idx, va_idx in skf.split(Xtr, y_train):
            clf = LogisticRegression(C=C, max_iter=PARAMS.lr_max_iter,
                                     class_weight="balanced",
                                     solver="lbfgs", random_state=SEED)
            clf.fit(Xtr[tr_idx], y_train[tr_idx])
            cv_aucs.append(roc_auc_score(
                y_train[va_idx],
                clf.predict_proba(Xtr[va_idx])[:, 1]))
        m = float(np.mean(cv_aucs))
        if m > best_cv:
            best_cv = m
            best_C = C
    clf = LogisticRegression(C=best_C, max_iter=PARAMS.lr_max_iter,
                             class_weight="balanced",
                             solver="lbfgs", random_state=SEED)
    clf.fit(Xtr, y_train)
    proba = clf.predict_proba(Xte)[:, 1]
    auc = float(roc_auc_score(y_test, proba))
    precisions, recalls, thresholds = precision_recall_curve(y_test, proba)
    if len(thresholds):
        f1s = 2 * precisions[1:] * recalls[1:] / (precisions[1:] + recalls[1:] + 1e-12)
        idx = int(np.argmax(f1s))
        thr = float(thresholds[idx])
    else:
        thr = 0.5
    f1 = float(f1_score(y_test, (proba >= thr).astype(int), zero_division=0))

    rng = np.random.default_rng(SEED)
    n = len(y_test)
    aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        if len(np.unique(y_test[idx])) > 1:
            aucs.append(roc_auc_score(y_test[idx], proba[idx]))
    ci_lo = float(np.percentile(aucs, 2.5)) if aucs else auc
    ci_hi = float(np.percentile(aucs, 97.5)) if aucs else auc
    return dict(auc=auc, auc_ci_low=ci_lo, auc_ci_high=ci_hi,
                f1=f1, threshold=thr, best_C=float(best_C))


CFG = get_config()
print(f"Dataset: {CFG.name} | Device: {DEVICE}")

df_all = load_disease_labels(CFG, ["cardiomegaly"])
rng = np.random.default_rng(SEED)
n = min(N_TARGET, len(df_all))
idx = rng.choice(len(df_all), size=n, replace=False)
df = df_all.iloc[idx].reset_index(drop=True)
df["_stratum"] = "0"
train_df, test_df = stratified_split(df, "_stratum", test_frac=0.2, seed=SEED)
print(f"train={len(train_df)}  test={len(test_df)}")

model, _ = build_resnet50()

records = []
for pert_name, injector, patch_size in PERTURBATIONS:
    print(f"\n--- {pert_name} ---")
    Xc_tr, Xp_tr = extract_features(model, train_df, injector, patch_size)
    Xc_te, Xp_te = extract_features(model, test_df, injector, patch_size)
    Xtr = np.vstack([Xc_tr, Xp_tr])
    ytr = np.concatenate([np.zeros(len(Xc_tr)), np.ones(len(Xp_tr))]).astype(int)
    Xte = np.vstack([Xc_te, Xp_te])
    yte = np.concatenate([np.zeros(len(Xc_te)), np.ones(len(Xp_te))]).astype(int)
    res = fit_eval(Xtr, ytr, Xte, yte)
    print(f"  resnet50  AUC={res['auc']:.4f} "
          f"[{res['auc_ci_low']:.4f}, {res['auc_ci_high']:.4f}]")
    records.append(dict(
        dataset=CFG.dataset, perturbation=pert_name, mode="resnet50",
        auc=res["auc"], auc_ci_low=res["auc_ci_low"], auc_ci_high=res["auc_ci_high"],
        f1=res["f1"], threshold=res["threshold"], best_C=res["best_C"],
        n_features=int(Xtr.shape[1]),
        n_train=int(len(ytr)), n_test=int(len(yte)),
    ))

new_rows = pd.DataFrame(records)
exp07_dir = WORK / f"v4_exp07_rawpixel_baseline_{CFG.dataset}"
exp07_dir.mkdir(parents=True, exist_ok=True)
existing = exp07_dir / f"exp07_{CFG.dataset}_rawpixel_baseline.parquet"
if existing.exists():
    df_existing = pd.read_parquet(existing)
    # Drop any prior resnet50 rows for these perturbations (idempotent re-run).
    drop_mask = ((df_existing["mode"] == "resnet50")
                 & (df_existing["perturbation"].isin(
                     [r["perturbation"] for r in records])))
    df_existing = df_existing[~drop_mask]
    merged = pd.concat([df_existing, new_rows], ignore_index=True)
else:
    merged = new_rows
merged.to_parquet(existing, index=False)
print(f"\nWrote {len(new_rows)} new rows -> {existing}")
print(merged[merged["mode"] == "resnet50"][
    ["perturbation", "mode", "auc", "auc_ci_low", "auc_ci_high"]].to_string())
